In [3]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

# --- UTILITY: QUANTUM RANDOM GENERATOR ---
def get_quantum_random_bits(n):
    bits = []
    backend = BasicSimulator()
    for _ in range(n):
        qc = QuantumCircuit(1)
        qc.h(0)
        qc.measure_all()
        t_qc = transpile(qc, backend)
        result = backend.run(t_qc, shots=1).result()
        bits.append(int(list(result.get_counts().keys())[0]))
    return bits

n_qubits = 40 
threshold = 0.1 

# --- ALICE: PREPARATION ---
alice_bits = get_quantum_random_bits(n_qubits)
alice_bases = get_quantum_random_bits(n_qubits)

# --- EVE: THE ATTACKER ---
eve_bases = get_quantum_random_bits(n_qubits)
eve_results = []
intercepted_qubits = []
backend = BasicSimulator()

for i in range(n_qubits):
    qc = QuantumCircuit(1)
    if alice_bits[i] == 1: qc.x(0)
    if alice_bases[i] == 1: qc.h(0)
    
    if eve_bases[i] == 1: qc.h(0)
    qc.measure_all()
    
    t_qc = transpile(qc, backend)
    m_result = int(list(backend.run(t_qc, shots=1).result().get_counts().keys())[0])
    
    resend_qc = QuantumCircuit(1)
    if m_result == 1: resend_qc.x(0)
    if eve_bases[i] == 1: resend_qc.h(0)
    intercepted_qubits.append(resend_qc)

# --- BOB: MEASUREMENT ---
bob_bases = get_quantum_random_bits(n_qubits)
bob_results = []

for i in range(n_qubits):
    qc = intercepted_qubits[i]
    if bob_bases[i] == 1: qc.h(0)
    qc.measure_all()
    t_qc = transpile(qc, backend)
    bob_results.append(int(list(backend.run(t_qc, shots=1).result().get_counts().keys())[0]))

# --- SIFTING & DETECTION ---
alice_key = []
bob_key = []
for i in range(n_qubits):
    if alice_bases[i] == bob_bases[i]:
        alice_key.append(alice_bits[i])
        bob_key.append(bob_results[i])

errors = sum(1 for a, b in zip(alice_key, bob_key) if a != b)
error_rate = errors / len(alice_key) if len(alice_key) > 0 else 0

print(f"Error Rate with Attacker: {error_rate * 100:.2f}%")
if error_rate > threshold:
    print(f"ATTACK DETECTED! Error rate {error_rate*100}% is above threshold.") 

Error Rate with Attacker: 26.92%
ATTACK DETECTED! Error rate 26.923076923076923% is above threshold.
